# Building a Movie Recommendation System
## Milestone 2: Honest Testing and Baselines to Beat
*MATH 344L Applied Linear Algebra Lab*

Goal. Before building a recommender with linear algebra, we need a fair way to tell whether it
works. We hide some ratings, train every model only on the ratings that remain, and score the
predictions only on the hidden ratings. We also build simple average based recommenders. Their
scores are the baselines that the SVD model in Milestone 3 has to beat.

Error is measured with RMSE, and a smaller RMSE means more accurate predictions.

---
## Part A: Load the MovieLens ratings

In [1]:
import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"])

print(ratings.head())
print("Number of ratings:", len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


This cell imports the tools we need and downloads MovieLens 100K, but only if the ml-100k
folder is not already there. Then pd.read_csv reads the ratings file into a table.
Each row is one known rating, made of a user, a movie, a number of stars, and a timestamp. There
are 100,000 rows in total. This is called long form, and later we turn the training part of it
into a user by movie matrix.

---
## Part B: Make an honest train/test split

In [ ]:
def split_ratings(ratings, test_frac=0.10, seed=0):
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=bool)

    user_left = ratings.groupby("user").size().to_dict()
    item_left = ratings.groupby("item").size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row["user"], row["item"]

        if (rng.random() < test_frac
                and user_left[user] > 1
                and item_left[item] > 1):
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)
print("train:", len(train), "test:", len(test))
print("test fraction:", round(len(test) / len(ratings), 3))

train: 90032 test: 9968
test fraction: 0.1


The function goes through the ratings in a random order. Each rating gets about a 10% chance of
being hidden in the test set. A rating is only hidden if its user and its movie both still have
at least one other rating left in training, so no user or movie disappears from the training data.
is_test is a True/False array marking the hidden ratings, and is_test flips it to pick out
the training ratings.

We got 90,032 training ratings and 9,968 test ratings, which is almost exactly 10%. Fixing the
seed makes the split the same every time we run it, so every model is trained and tested on the
same ratings and the comparison is fair.

The test ratings must stay hidden from every step that builds a model. If a test rating is used to
compute an average or an SVD, the score looks better than it really is. This mistake is called
data leakage.

---
## Part C: Build the training matrix

In [ ]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
    """Return a user by movie matrix. Missing ratings are np.nan."""
    R = np.full((N_USERS, N_ITEMS), np.nan)
    R[df["user"].to_numpy() - 1,
      df["item"].to_numpy() - 1] = df["rating"].to_numpy()
    return R

R = to_matrix(train)
known = ~np.isnan(R)

print("Matrix shape:", R.shape)
print("Known training ratings:", known.sum())

Matrix shape: (943, 1682)
Known training ratings: 90032


to_matrix starts with a 943 by 1682 matrix full of np.nan and then fills in each training
rating at its user's row and movie's column. User and movie IDs start at 1 but Python positions
start at 0, which is why the code subtracts 1. A missing rating is stored as np.nan, meaning
unknown, and not as zero stars. known is True exactly where R has a training rating, and its
sum matches the 90,032 training ratings.

In [4]:
truth = test["rating"].to_numpy()
u = test["user"].to_numpy() - 1
i = test["item"].to_numpy() - 1

For each hidden test rating, truth holds the real number of stars, and u and i hold the row
and column of that rating in the matrix. Every model's predictions get compared to truth.

---
## Part D: Score predictions with RMSE

In [5]:
def rmse(pred, truth):
    """Root mean squared error, measured in stars."""
    return np.sqrt(np.mean((pred - truth)**2))

RMSE takes the error for each hidden rating, squares it, averages the squares, and takes the
square root.

$$\text{RMSE} = \sqrt{\frac{1}{N}\sum_{j=1}^{N} (\text{prediction}_j - \text{truth}_j)^2}$$

Because of the square root, RMSE is measured in stars. An RMSE near 1 means predictions are
usually about one star off. Lower is better.

---
## Part E: Build simple baselines

In [6]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)

n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0
item_mean[has_rating] = (
    np.nansum(R, axis=0)[has_rating] / n_item[has_rating])

print("Overall average:", round(mu, 3))

Overall average: 3.53


np.nanmean takes an average while skipping the np.nan entries. With no axis, mu is the
average of every training rating, about 3.53 stars. With axis=1 it averages across each row, so
user_mean has one average per user. For movies, known.sum(axis=0) counts the training ratings
in each column and we divide the column sums by those counts. item_mean starts out filled with
mu, so a movie with no training ratings would just be predicted at the overall average.

In [7]:
scores = {}
scores["always 3"] = rmse(np.full(len(test), 3.0), truth)
scores["overall average"] = rmse(np.full(len(test), mu), truth)
scores["user average"] = rmse(user_mean[u], truth)
scores["movie average"] = rmse(item_mean[i], truth)

for name, score in scores.items():
    print(f"{name:16s} {score:.4f}")

always 3         1.2516
overall average  1.1349
user average     1.0426
movie average    1.0284


These are four simple ways to guess every hidden rating. "Always 3" guesses 3 stars for
everything. "Overall average" guesses about 3.53 for everything. "User average" looks up the
average of the person who made the rating (user_mean[u]), and "movie average" looks up the
average of the movie being rated (item_mean[i]).

Using the overall average beats always guessing 3 because real ratings lean positive. The movie
average does best of the four, since some movies are just liked a lot more than others.

---
## Part F: Combine user and movie effects

Some users rate generously and some movies are widely liked. The bias baseline uses both effects.

$$\text{prediction} = \mu + b_{\text{user}} + b_{\text{item}}$$

In [8]:
def bias_baseline(R):
    """Overall mean plus one user effect and one movie effect."""
    known = ~np.isnan(R)
    mu = np.nanmean(R)

    b_user = np.nanmean(R - mu, axis=1)
    residual = R - mu - b_user[:, None]

    n_item = known.sum(axis=0)
    b_item = np.zeros(R.shape[1])
    has_rating = n_item > 0
    b_item[has_rating] = (
        np.nansum(residual, axis=0)[has_rating] / n_item[has_rating])

    return mu, b_user, b_item

b_user is how far each user's average is from the overall mean, so a generous rater gets a
positive value. b_user[:, None] turns it into a column so that NumPy subtracts each user's
effect across that user's whole row. What is left over is the residual, and b_item is each
movie's average residual. That is how much better or worse a movie does than expected once we
account for who rated it. A movie with no training ratings gets an effect of 0.

In [9]:
mu, b_user, b_item = bias_baseline(R)
pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)

scores["user + movie effects"] = rmse(pred, truth)
print(pd.Series(scores, name="test RMSE").round(4).sort_values())

user + movie effects    0.9545
movie average           1.0284
user average            1.0426
overall average         1.1349
always 3                1.2516
Name: test RMSE, dtype: float64


The prediction for each hidden rating is the overall mean plus that user's effect plus that
movie's effect. np.clip keeps every prediction between 1 and 5 stars. Combining both effects
gives the lowest RMSE, about 0.95, beating every single average. This is a simple one pass
baseline and not a finished recommender, but it is the score the SVD model needs to beat in
Milestone 3.

---
## Practice Exercises

### Exercise 1: By hand, then by computer

$$\begin{bmatrix} 5 & 3 & ? \\ 4 & ? & 1 \\ ? & 2 & 2 \end{bmatrix}$$

Overall mean. The six known ratings add up to 5 + 3 + 4 + 1 + 2 + 2 = 17, so
$\mu = 17/6 \approx 2.833$.

User effects. Each user's effect is their average minus $\mu$.
- User 1 averages (5 + 3)/2 = 4, so $b_1 = 4 - 17/6 = 7/6 \approx 1.167$
- User 2 averages (4 + 1)/2 = 2.5, so $b_2 = 2.5 - 17/6 = -1/3 \approx -0.333$
- User 3 averages (2 + 2)/2 = 2, so $b_3 = 2 - 17/6 = -5/6 \approx -0.833$

Movie effects. Subtracting $\mu + b_{\text{user}}$ from a rating is the same as subtracting that
user's average, which gives these residuals.

$$\begin{bmatrix} 1 & -1 & ? \\ 1.5 & ? & -1.5 \\ ? & 0 & 0 \end{bmatrix}$$

Each movie's effect is the average of its column.
- Movie 1 is (1 + 1.5)/2 = 1.25
- Movie 2 is (-1 + 0)/2 = -0.5
- Movie 3 is (-1.5 + 0)/2 = -0.75

Now the same thing by computer.

In [10]:
tiny = np.array([
    [5, 3, np.nan],
    [4, np.nan, 1],
    [np.nan, 2, 2],
])

mu_t, b_user_t, b_item_t = bias_baseline(tiny)
print("mu:", round(mu_t, 4))
print("b_user:", b_user_t.round(4))
print("b_item:", b_item_t.round(4))

mu: 2.8333
b_user: [ 1.1667 -0.3333 -0.8333]
b_item: [ 1.25 -0.5  -0.75]


The computer gives the same values as the hand calculation. Using them, the three missing
ratings would be predicted as user 1 on movie 3 at 4 - 0.75 = 3.25, user 2 on movie 2 at
2.5 - 0.5 = 2.0, and user 3 on movie 1 at 2 + 1.25 = 3.25.

### Exercise 2: See what cheating looks like

In [11]:
R_all = to_matrix(ratings)
cheat_user_mean = np.nanmean(R_all, axis=1)

cheat_score = rmse(cheat_user_mean[u], truth)
print("Honest user average RMSE:  ", round(scores["user average"], 4))
print("Cheating user average RMSE:", round(cheat_score, 4))

Honest user average RMSE:   1.0426
Cheating user average RMSE: 1.0322


Yes, the cheating RMSE is lower, about 1.032 instead of 1.043. The matrix built from all ratings
includes the hidden test ratings, so each user's average already contains the exact ratings we are
trying to predict. The model gets to peek at the answers. That score is misleading because a real
recommender never knows a rating before the user gives it, so it would not do that well on new
ratings. This is data leakage, and it makes a model look better than it really is.

### Exercise 3: Is it luck?

In [12]:
def all_scores(seed):
    """Split with the given seed and return the test RMSE of all five baselines."""
    train, test = split_ratings(ratings, seed=seed)
    R = to_matrix(train)
    known = ~np.isnan(R)
    truth = test["rating"].to_numpy()
    u = test["user"].to_numpy() - 1
    i = test["item"].to_numpy() - 1

    mu = np.nanmean(R)
    user_mean = np.nanmean(R, axis=1)
    n_item = known.sum(axis=0)
    item_mean = np.full(N_ITEMS, mu)
    has_rating = n_item > 0
    item_mean[has_rating] = np.nansum(R, axis=0)[has_rating] / n_item[has_rating]

    out = {}
    out["always 3"] = rmse(np.full(len(test), 3.0), truth)
    out["overall average"] = rmse(np.full(len(test), mu), truth)
    out["user average"] = rmse(user_mean[u], truth)
    out["movie average"] = rmse(item_mean[i], truth)
    m, bu, bi = bias_baseline(R)
    out["user + movie effects"] = rmse(np.clip(m + bu[u] + bi[i], 1, 5), truth)
    return out

compare = pd.DataFrame({"seed 0": all_scores(0), "seed 1": all_scores(1)})
compare["change"] = compare["seed 1"] - compare["seed 0"]
compare.sort_values("seed 0").round(4)

,seed 0,seed 1,change
user + movie effects,0.9545,0.9577,0.0032
movie average,1.0284,1.0159,-0.0125
user average,1.0426,1.0345,-0.0081
overall average,1.1349,1.1127,-0.0222
always 3,1.2516,1.2336,-0.0180


The values do change with seed 1, by up to about 0.02 stars, because a different 10% of the
ratings is hidden and some hidden sets are easier to predict than others. The ranking does not
change though. User and movie effects is best, then movie average, user average, overall average,
and always 3, for both seeds.

This means a small difference between two recommenders, like 0.01, could just be luck from the
split. To compare two models fairly they have to be scored on the same test ratings, and it helps
to check a few different seeds. A model is only really better if it wins on all of them, which
these baselines do here.